In [1]:
try:
    %load_ext autoreload --quiet
except ImportError:
    %reload_ext autoreload
%autoreload 2

import hashlib
from typing import List, Any
from flow_merge.lib.snapshot.data_architecture._normalized_slices import NormalizedSlice
from pydantic import BaseModel, computed_field, Field
import datetime
from pathlib import Path
import json

class MergePlan(BaseModel):
    created_at: datetime.datetime = Field(default=datetime.datetime.now())
    base_model: str
    tokenizer_mode: str
    tokenizer_interpolation_method: str
    slices: List[NormalizedSlice]
    lib_version: str

    @classmethod
    def from_config(cls, config: Any) -> "MergePlan":
        return cls(
            created_at=datetime.datetime.now(),
            base_model=config.base_model,
            tokenizer_mode=config.tokenizer_mode,
            tokenizer_interpolation_method=config.tokenizer_interpolation_method,
            slices=config.slices,
            lib_version=config.lib_version,
        )

    @classmethod
    def from_file(cls, file_path: Path | str) -> "MergePlan":
        with open(Path(file_path).resolve(), "rb") as f:
            parsed = json.load(f)
            return cls(**parsed)

    @computed_field
    @property
    def sha(self) -> str:
        obj = self.model_dump_json(exclude={"created_at", "sha"}).encode("utf-8")
        return hashlib.md5(obj).hexdigest()

mp = MergePlan.from_file("../flow_merge/lib/example-slices.json")


In [2]:
print(mp.sha)

dacbc64dcbfa795dc7eb9a2c267bf051


In [7]:
from flow_merge.lib.model import Model
from operator import concat
from functools import reduce



# get models from all the slices
all_model_path_occurrences: List[str|None] = [model_field.model for x in mp.slices for model_field in x.sources]
all_distinct_model_paths: List[str|None] = list({model for model in all_model_path_occurrences})
all_model_objects: List[Model] = [Model.from_path(model_path) for model_path in all_distinct_model_paths]

# construct merged tokenizer out of all the distinct models



# Plan -> Stage (where List[NormalizedSlice] -> List[ExecutableSlice])

# Shard loaded and tensors retrieved in execution -> Shard flushed (in the future we can download only the shards that are needed)


# id: ModelId
# path: Path
# metadata: ModelMetadata
# file_to_tensor_index: Optional[Dict]
# shards: List[ShardFile]
# architecture: ModelArchitecture

# get merge method from method config
# base_model: Model,
# task_base_model_weight: ModelWeight,
# task_models_with_weights: Dict[Model, ModelWeight],
# tokenizer: Tokenizer,
# method_config,
# sources



['Qwen/Qwen1.5-0.5B']
Loading Model from path: from_path
Creating metadata: _create_metadata
Loading Model Info: load_model_info
Fetching model info from HF: fetch_hf_model_info
Creating File Metadata_list from HF: create_file_metadata_list_from_hf
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Missing model.safetensors.index.json file
Missing pytorch_model .bin files
Missing pytorch_model.bin.index.json file
Missing adapter files
Index files not found, using single shard file fallback.
Creating architecture: _create_architecture
Substitute the layer templates
Generating model weights: _generate_model_weights
Creatin